## Dataset Preparation
Feature and Target Sepearation

In [26]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline

words = open('../data/names.txt', 'r').read().splitlines()

In [27]:
chars = sorted(set(''.join(words)))

stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {s:i for i,s in stoi.items()}

# itos[1] = 'a'
# stoi['a'] = 1

In [43]:
CONTEXT_SIZE = 3

X = []
y = []
for w in words[:5]:
    # print(w)
    context = [0] * CONTEXT_SIZE
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        y.append(ix)

        context = context[1:] + [ix]
        # print(''.join(itos[i] for i in context), '--->', itos[ix])


# X is [x1, x2, x3, ... xN] 
X = torch.tensor(X)
y = torch.tensor(y)

In [44]:
DIM_SIZE = 2

# C is an embedded dimension
C = torch.randn((27, DIM_SIZE))

# OHE, but we do it algebraically (it's the same zero cancel out effect)
emb = C[X]

print(X.shape)
print(C.shape)
print(emb.shape)

torch.Size([32, 3])
torch.Size([27, 2])
torch.Size([32, 3, 2])


In [47]:
W1 = torch.randn((CONTEXT_SIZE * DIM_SIZE, 100))
B1 = torch.randn(100)

h = torch.tanh(emb.view(-1, CONTEXT_SIZE * DIM_SIZE) @ W1 + B1)
h.shape

torch.Size([32, 100])

In [51]:
W2 = torch.randn((100, 27))
B2 = torch.randn(27)

logits = h @ W2 + B2
counts = logits.exp()
probs = counts/counts.sum(1, keepdims=True)


## Unified Version

In [53]:
g  = torch.Generator().manual_seed(2147483647)
C  = torch.randn((27, DIM_SIZE), generator=g) 
W1 = torch.randn((CONTEXT_SIZE * DIM_SIZE, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True

In [122]:
import torch.nn.functional as F

epochs = 100
batch  = 64
lr = 0.1
for _ in range(epochs):
    # forward pass
    ix = torch.randint(0, X.shape[0], (batch,))

    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, CONTEXT_SIZE * DIM_SIZE) @ W1 + b1)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data -= lr * p.grad

    print(loss.item())

0.31158527731895447
0.14888598024845123
0.3166160583496094
0.352116197347641
0.27274537086486816
0.28455018997192383
0.28352510929107666
0.2713611125946045
0.3052669167518616
0.14860522747039795
0.38365915417671204
0.14300094544887543
0.3453139364719391
0.22601939737796783
0.2937068045139313
0.20611397922039032
0.2617192566394806
0.18859340250492096
0.3490932285785675
0.42516347765922546
0.25681087374687195
0.29943081736564636
0.28193798661231995
0.1976657658815384
0.2899463474750519
0.20946060121059418
0.16260379552841187
0.2778465747833252
0.4194823205471039
0.3435439169406891
0.2969052791595459
0.18171221017837524
0.25400739908218384
0.427920937538147
0.15980781614780426
0.19751641154289246
0.38995781540870667
0.24578264355659485
0.47850096225738525
0.23742994666099548
0.363599956035614
0.3687713146209717
0.24694137275218964
0.1482606828212738
0.19169627130031586
0.35914936661720276
0.16884304583072662
0.23265667259693146
0.25426405668258667
0.3160320520401001
0.4938768446445465
0.1

In [123]:
emb = C[X]
h = torch.tanh(emb.view(-1, CONTEXT_SIZE * DIM_SIZE) @ W1 + b1)
logits = h @ W2 + b2

loss = F.cross_entropy(logits, y)

loss.item()

0.2580711841583252

KeyboardInterrupt: 